# NBA 3.1 - 31 Competitor drugs NBRx share (per cohort)

Upgrade of NBA 3.0 `03_Feature_Engineering/03_HCP_Level_Features/31_Competitor_drugs_NBRX_share`. Spec: `Feature Analysis/92_competitor_comorbidity_nba31_readiness_plan.md` Phase 2 (CP1-CP7) and `10_competitor_nbrx_share_features_nba31_changes.md`. Plain-language summary: `93_competitor_comorbidity_plan_in_simple_words.md`.

**What changes vs NBA 3.0**

| NBA 3.0 | NBA 3.1 |
| --- | --- |
| LRx V6, all indication groups mixed | LRx V7, two scopes: `CKD` (`INDICATION_GROUP` contains CKD) and `ALL` (CP2) |
| Farxiga brand + generic merged (`LIKE '%FARXIGA%'`) | `FARXIGA_BRAND`, `FARXIGA_GENERIC` and combined `FARXIGA` (CP3) |
| 12-month window + running total | 3-month and 12-month windows, no running total (CP4) |
| SGLT2 only | + steroidal MRA in CKD scope (CP5), + GLP-1 in T2D and HF scopes (CP6) |
| Static build, `NPI_NUMBER` + `SVC_DT_M` (leakage) | One row per HCP x cohort, LRx up to the cohort's `lookback_end_date` month, `HCP_COHORT_ID` (CP1) |

**Columns** (W = 3 or 12)

| Group | Columns |
| --- | --- |
| SGLT2, scopes CKD and ALL | `TOTAL_NBRX_{FARXIGA, FARXIGA_BRAND, FARXIGA_GENERIC, JARDIANCE, SGLT2}_{S}_IN_LAST_{W}_MONTH`, `PERC_{FARXIGA, FARXIGA_BRAND, FARXIGA_GENERIC, JARDIANCE}_NBRX_SHARE_{S}_IN_LAST_{W}_MONTH` |
| CKD scope only (CP5, CP7) | `TOTAL_NBRX_{INVOKANA, STEGLATRO, STEROIDAL_MRA}_CKD_IN_LAST_{W}_MONTH` |
| GLP-1, scopes T2D and HF (CP6) | `TOTAL_NBRX_GLP1_{S}_IN_LAST_{W}_MONTH`, `PERC_GLP1_NBRX_SHARE_{S}_IN_LAST_{W}_MONTH` (GLP-1 / (GLP-1 + SGLT2)) |

**Business rules**
* SGLT2 denominator = Farxiga (brand + generic) + Jardiance + Invokana + Steglatro in the same scope and window (as NBA 3.0). Kerendia is the target and never a competitor feature.
* Steroidal MRA = V7 `PRODUCT_CLASS = 'sMRA'` (spironolactone, eplerenone and their brands). GLP-1 = semaglutide (Ozempic, Wegovy, Wegovy HD, Rybelsus) + tirzepatide (Mounjaro, Zepbound).
* `TOTAL_*` = 0 when the HCP has no NBRx; `PERC_*` is empty when the denominator is 0.
* Reading rule D8: a high CKD-scoped SGLT2 share is expected to be **positive** for Kerendia. Flag any model that learns the opposite.

Cohorts: `cohorts/lrx` when it exists, else the claims cohorts (`cohorts/lrx` did not exist on 2026-11-17). Output: `hcp_level_model/dev_pipeline/notebook_output/hcp_features/competitor_nbrx_share` (Delta). Widget `TEST_HCP_SAMPLE` > 0 writes to `competitor_nbrx_share_test`.

## Phase 0 validation status (plan 92, Section 3), checked 2026-11-17

**Score: 6 of 10 met, 3 partly met, 1 not met.** Checks that matter for this notebook are in **bold**.

| # | Validation | Result | Status |
| --- | --- | --- | --- |
| **V1** | **Indication coverage in LAAD** | **Share of NBRx with an indication group, monthly 2023-06 to 2026-09: SGLT2 91.5-93.4%, Entresto 100%, steroidal MRA 44.1-49.3%. Stable over time. CKD share of SGLT2 NBRx rose from 32.7% (first 6 months) to 42.5% (last 6 months)** | **Met.** `TOTAL_NBRX_STEROIDAL_MRA_CKD_*` sees only about half of MRA NBRx |
| V2 | CKD stage 3 codes present | Dx event list: 11,685,734 stage-3 patients. `Dx_03`: 25,167 | Met for 09, not met for `Dx_03` (affects 11) |
| V3 | HF LVEF >= 40% proxy sign-off | Medical review not done | Not met (not used here) |
| **V4** | **CKD indication groups in LRx** | **V7 groups: CKD, T2D + CKD, T2D + CKD + HF, CKD + HF. No CVD groups, unlike the plan's expected list** | **Met.** CKD scope = `INDICATION_GROUP` contains CKD |
| **V5** | **Farxiga brand vs generic labels** | **Only `FARXIGA BRAND` and `FARXIGA GENERIC`. Generic NBRx exists from 2023-11-03 (about 9-15k a month) and rises from Jan 2026 (43k), not from 2026-04-06** | **Partly met.** The split columns are valid; generic shares are non-zero before the LOE |
| V6 | Entresto generic | `Rx_01_new` generic from 2024-11-27; LRx generic from 2025-01-31, ramp Jul-Aug 2025 | Met |
| **V7** | **GLP-1 in LRx** | **12M NBRx 7.01M: Mounjaro 2.71M, Ozempic 1.84M, Zepbound 1.26M, Wegovy 0.95M, Rybelsus 0.23M, Wegovy HD 0.03M. 5.29M in T2D groups, 0.74M in HF groups** | **Met** |
| **V8** | **Parity vs NBA 3.0** | **Proxy only: `TOTAL_NBRX_{JARDIANCE,FARXIGA}_ALL_IN_LAST_12_MONTH` vs NBA 3.0 `{JARDIANCE,FARXIGA}_IN_LAST_12_MONTH` at the same `SVC_DT_M` (294,598 shared HCPs). Correlation: Jardiance 0.71 (c1) to 0.93 (c28), Farxiga 0.84-0.93. Jardiance totals about 20% higher in 3.1** | **Partly met.** Differences come from V7 vs V6 and the drug definitions; no full 3.0 rebuild per cohort |
| V9 | Rows after each event | Entresto LOE: 14 cohorts, 86,065 writers. Farxiga LOE: 5 cohorts, 32,731 writers | Met |
| V10 | Entresto prescriber specialty mix | Brand writers: CARD 16.1%, NEPH 2.6%. Kerendia writers: NEPH 19.0%, CARD 3.4% | Met (answered) |

**For this notebook:** V1, V4 and V7 pass. V5 and V8 are partly met; neither blocks the build.

In [0]:
import os, json, builtins
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0 = full universe)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe_without_call_hcps"
LRX_TABLE = "`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7"
HCP_FEATURES_OUT_31 = f"{VOL31}/hcp_level_model/dev_pipeline/notebook_output/hcp_features"
OUTPUT_PATH = f"{HCP_FEATURES_OUT_31}/competitor_nbrx_share" + ("_test" if TEST_HCP_SAMPLE else "")
assert "kerendia_nba_3_1_volume" in OUTPUT_PATH

# LRx cohorts if they exist, otherwise the claims cohorts
COHORTS_DIR = f"{VOL31}/cohorts/lrx" if os.path.exists(f"{VOL31}/cohorts/lrx/metadata.json") else f"{VOL31}/cohorts/claims"

# V7 PRODUCT_GROUP -> drug key
DRUGS = {
    "FARXIGA_BRAND":   ["FARXIGA BRAND"],
    "FARXIGA_GENERIC": ["FARXIGA GENERIC"],
    "JARDIANCE":       ["JARDIANCE"],
    "INVOKANA":        ["INVOKANA"],
    "STEGLATRO":       ["STEGLATRO"],
    "GLP1":            ["OZEMPIC", "WEGOVY", "WEGOVY HD", "RYBELSUS", "MOUNJARO", "ZEPBOUND"],
}
SMRA_CLASS = "sMRA"          # STEROIDAL_MRA = every V7 product with PRODUCT_CLASS sMRA
SCOPES = {
    "ALL": F.lit(True),
    "CKD": F.upper(F.col("INDICATION_GROUP")).contains("CKD"),
    "T2D": F.upper(F.col("INDICATION_GROUP")).contains("T2D"),
    "HF":  F.upper(F.col("INDICATION_GROUP")).contains("HF"),
}
WINDOWS = [3, 12]
print(f"Cohorts from {COHORTS_DIR} | output {OUTPUT_PATH}")

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert len(cohorts) == meta["cohort_count"], meta

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string")).distinct()
if TEST_HCP_SAMPLE:
    hcp_universe = hcp_universe.orderBy(F.xxhash64("PROVIDER_ID")).limit(TEST_HCP_SAMPLE)
hcp_universe = hcp_universe.cache()
n_hcp = hcp_universe.count()

coh = (spark.createDataFrame(cohorts[["cohort_id", "lookback_end_date"]].astype(str))
       .withColumn("SVC_DT_M", F.to_date("lookback_end_date")).drop("lookback_end_date"))
END_MONTHS = [pd.to_datetime(d).date() for d in cohorts["lookback_end_date"]]
FIRST_MONTH = (pd.to_datetime(min(END_MONTHS)) - pd.DateOffset(months=max(WINDOWS) - 1)).date()
print(f"{meta.get('feature_family')} cohorts: {len(cohorts)}, lookback end {min(END_MONTHS)}..{max(END_MONTHS)} | LRx months {FIRST_MONTH}..{max(END_MONTHS)} | HCPs {n_hcp:,}")

In [0]:
pg_to_drug = F.create_map(*[x for d, pgs in DRUGS.items() for p in pgs for x in (F.lit(p), F.lit(d))])
lrx = (spark.table(LRX_TABLE)
       .withColumn("M", F.trunc("PERIOD_ID", "month"))
       .filter(F.col("PROVIDER_ID").isNotNull()
               & (F.col("M") >= F.lit(str(FIRST_MONTH)).cast("date"))
               & (F.col("M") <= F.lit(str(max(END_MONTHS))).cast("date")))
       .withColumn("DRUG", F.when(F.col("PRODUCT_CLASS") == SMRA_CLASS, F.lit("STEROIDAL_MRA"))
                           .otherwise(pg_to_drug[F.upper(F.trim(F.col("PRODUCT_GROUP")))]))
       .filter(F.col("DRUG").isNotNull())
       .join(F.broadcast(hcp_universe), "PROVIDER_ID", "left_semi"))

# one column per drug x scope, monthly NBRx
monthly = lrx.groupBy("PROVIDER_ID", "M").agg(*[
    F.sum(F.when((F.col("DRUG") == d) & cond, F.col("NBRX")).otherwise(0)).alias(f"{d}__{s}")
    for d in list(DRUGS) + ["STEROIDAL_MRA"] for s, cond in SCOPES.items()])

# V1: share of NBRx without a usable indication group, by drug (last 12 months)
v1 = (lrx.filter(F.col("M") >= F.add_months(F.lit(str(max(END_MONTHS))).cast("date"), -11))
         .groupBy("DRUG").agg(F.sum("NBRX").alias("nbrx"),
              F.sum(F.when(F.col("INDICATION_GROUP").isNull() | (F.upper("INDICATION_GROUP") == "OTHERS"), F.col("NBRX")).otherwise(0)).alias("nbrx_others"),
              F.sum(F.when(SCOPES["CKD"], F.col("NBRX")).otherwise(0)).alias("nbrx_ckd"))
         .withColumn("pct_others", F.round(100 * F.col("nbrx_others") / F.col("nbrx"), 1))
         .withColumn("pct_ckd", F.round(100 * F.col("nbrx_ckd") / F.col("nbrx"), 1)).orderBy("DRUG"))
display(v1)

In [0]:
def tot(d, s, w):  return f"TOTAL_NBRX_{d}_{s}_IN_LAST_{w}_MONTH"
def perc(d, s, w): return f"PERC_{d}_NBRX_SHARE_{s}_IN_LAST_{w}_MONTH"
m = lambda d, s: F.col(f"{d}__{s}")

# window sums: monthly rows inside (lookback_end - W + 1 .. lookback_end]
win = monthly.join(F.broadcast(coh), (F.col("M") <= F.col("SVC_DT_M")) & (F.col("M") >= F.add_months(F.col("SVC_DT_M"), -(max(WINDOWS) - 1))))
sum_cols = [c for c in monthly.columns if "__" in c]
agg = win.groupBy("PROVIDER_ID", "cohort_id").agg(*[
    F.sum(F.when(F.col("M") >= F.add_months(F.col("SVC_DT_M"), -(w - 1)), F.col(c)).otherwise(0)).alias(f"{c}__{w}")
    for c in sum_cols for w in WINDOWS])
g = lambda d, s, w: F.coalesce(F.col(f"{d}__{s}__{w}"), F.lit(0))

sel, FEATURE_COLS = [], []
def add(expr, name):
    sel.append(expr.alias(name)); FEATURE_COLS.append(name)

for s in ["CKD", "ALL"]:
    for w in WINDOWS:
        farx = g("FARXIGA_BRAND", s, w) + g("FARXIGA_GENERIC", s, w)
        sglt2 = farx + g("JARDIANCE", s, w) + g("INVOKANA", s, w) + g("STEGLATRO", s, w)
        nums = {"FARXIGA": farx, "FARXIGA_BRAND": g("FARXIGA_BRAND", s, w),
                "FARXIGA_GENERIC": g("FARXIGA_GENERIC", s, w), "JARDIANCE": g("JARDIANCE", s, w)}
        for d, e in nums.items():
            add(e, tot(d, s, w))
        add(sglt2, tot("SGLT2", s, w))
        for d, e in nums.items():
            add(F.when(sglt2 > 0, e / sglt2), perc(d, s, w))
for w in WINDOWS:
    for d in ["INVOKANA", "STEGLATRO", "STEROIDAL_MRA"]:
        add(g(d, "CKD", w), tot(d, "CKD", w))
for s in ["T2D", "HF"]:
    for w in WINDOWS:
        sglt2 = sum([g(d, s, w) for d in ["FARXIGA_BRAND", "FARXIGA_GENERIC", "JARDIANCE", "INVOKANA", "STEGLATRO"]], F.lit(0))
        add(g("GLP1", s, w), tot("GLP1", s, w))
        add(F.when((g("GLP1", s, w) + sglt2) > 0, g("GLP1", s, w) / (g("GLP1", s, w) + sglt2)), perc("GLP1", s, w))
print(len(FEATURE_COLS), "features")

feat = agg.select("PROVIDER_ID", "cohort_id", *sel)
tot_cols = [c for c in FEATURE_COLS if c.startswith("TOTAL_")]
competitor_nbrx_share = (hcp_universe.crossJoin(F.broadcast(coh))
    .join(feat, ["PROVIDER_ID", "cohort_id"], "left")
    .fillna(0, subset=tot_cols)
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID",
            *[F.col(c).cast("double").alias(c) for c in FEATURE_COLS], "PULL_DT"))

os.makedirs(HCP_FEATURES_OUT_31, exist_ok=True)
competitor_nbrx_share.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH)
out = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {out.count():,} rows, {len(out.columns)} columns -> {OUTPUT_PATH}")

In [0]:
n_rows = out.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert out.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
perc_cols = [c for c in FEATURE_COLS if c.startswith("PERC_")]
checks = {
    "PERC outside [0,1]": " OR ".join(f"({c} < 0 OR {c} > 1)" for c in perc_cols),
    "TOTAL null/negative": " OR ".join(f"({c} < 0 OR {c} IS NULL)" for c in tot_cols),
    "FARXIGA != BRAND + GENERIC": " OR ".join(f"abs({tot('FARXIGA', s, w)} - {tot('FARXIGA_BRAND', s, w)} - {tot('FARXIGA_GENERIC', s, w)}) > 1e-6" for s in ['CKD', 'ALL'] for w in WINDOWS),
    "CKD > ALL": " OR ".join(f"{tot(d, 'CKD', w)} > {tot(d, 'ALL', w)} + 1e-6" for d in ['FARXIGA', 'JARDIANCE', 'SGLT2'] for w in WINDOWS),
    "3M > 12M": " OR ".join(f"{tot(d, s, 3)} > {tot(d, s, 12)} + 1e-6" for d in ['FARXIGA', 'JARDIANCE', 'SGLT2'] for s in ['CKD', 'ALL']),
}
res = {k: out.filter(v).count() for k, v in checks.items()}
print(res)
assert all(v == 0 for v in res.values())

# Independent check: Jardiance CKD 12M for the last cohort, straight from the LRx table with PERIOD_ID date bounds
last = cohorts.iloc[-1]
end = F.last_day(F.lit(str(last.lookback_end_date)[:10]).cast("date"))
start = F.add_months(F.trunc(end, "month"), -11)
exp = (spark.table(LRX_TABLE)
       .filter((F.col("PERIOD_ID") >= start) & (F.col("PERIOD_ID") <= end)
               & (F.upper("PRODUCT_GROUP") == "JARDIANCE") & F.upper("INDICATION_GROUP").contains("CKD"))
       .groupBy("PROVIDER_ID").agg(F.sum("NBRX").alias("exp")))
cmp = (out.filter(F.col("cohort_id") == last.cohort_id).join(exp, "PROVIDER_ID", "left").fillna(0, subset=["exp"]))
mism = cmp.filter(F.abs(F.col(tot("JARDIANCE", "CKD", 12)) - F.col("exp")) > 1e-6).count()
print(f"{last.cohort_id}: Jardiance CKD 12M mismatches vs direct recompute = {mism}")
assert mism == 0

In [0]:
last_df = out.filter(F.col("cohort_id") == cohorts.cohort_id.iloc[-1])
stats = last_df.agg(*[F.count(F.when(F.col(c) > 0, 1)).alias(f"nz_{c}") for c in FEATURE_COLS],
                    *[F.percentile_approx(c, 0.5).alias(f"med_{c}") for c in FEATURE_COLS]).collect()[0]
display(pd.DataFrame({"feature": FEATURE_COLS,
                      "pct_hcps_non_zero": [builtins.round(100 * stats[f"nz_{c}"] / n_hcp, 1) for c in FEATURE_COLS],
                      "median": [stats[f"med_{c}"] for c in FEATURE_COLS]}))

In [0]:
# Output table: one row per HCP x cohort (key HCP_COHORT_ID)
# Reads the saved output, so this cell and the next only need 'Paths and parameters' and 'Load cohorts and HCP universe' run first
out = spark.read.format("delta").load(OUTPUT_PATH)
FEATURE_COLS = [c for c in out.columns if c not in {"PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", "PULL_DT"}]
display(out.select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID", *FEATURE_COLS))

In [0]:
# Fill rate per feature: % of rows with a value (not null) and % with a non-zero value,
# over all cohorts and for the latest cohort
latest = cohorts.cohort_id.iloc[-1]

def _fill(df):
    n = df.count()
    r = df.agg(*[F.count(c).alias(f"nn_{c}") for c in FEATURE_COLS],
               *[F.count(F.when(F.col(c) != 0, 1)).alias(f"nz_{c}") for c in FEATURE_COLS]).collect()[0]
    return ([builtins.round(100 * r[f"nn_{c}"] / n, 2) for c in FEATURE_COLS],
            [builtins.round(100 * r[f"nz_{c}"] / n, 2) for c in FEATURE_COLS])

nn_all, nz_all = _fill(out)
nn_last, nz_last = _fill(out.filter(F.col("cohort_id") == latest))
display(pd.DataFrame({"feature": FEATURE_COLS,
                      "pct_filled_all_cohorts": nn_all, "pct_non_zero_all_cohorts": nz_all,
                      f"pct_filled_{latest}": nn_last, f"pct_non_zero_{latest}": nz_last}))

## Feature dictionary, business rules and business logic

**Features created (50).** `TOTAL_*` = sum of NBRx; `PERC_*` = drug NBRx / SGLT2 NBRx (GLP-1: GLP-1 / (GLP-1 + SGLT2)) in the same scope and window. W = last 3 or 12 months.

| Group | Features | Count |
| --- | --- | --- |
| SGLT2 counts, CKD scope | `TOTAL_NBRX_FARXIGA_CKD_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_FARXIGA_BRAND_CKD_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_FARXIGA_GENERIC_CKD_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_JARDIANCE_CKD_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_SGLT2_CKD_IN_LAST_{3,12}_MONTH` | 10 |
| SGLT2 shares, CKD scope | `PERC_FARXIGA_NBRX_SHARE_CKD_IN_LAST_{3,12}_MONTH`, `PERC_FARXIGA_BRAND_NBRX_SHARE_CKD_IN_LAST_{3,12}_MONTH`, `PERC_FARXIGA_GENERIC_NBRX_SHARE_CKD_IN_LAST_{3,12}_MONTH`, `PERC_JARDIANCE_NBRX_SHARE_CKD_IN_LAST_{3,12}_MONTH` | 8 |
| SGLT2 counts, all indications | `TOTAL_NBRX_FARXIGA_ALL_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_FARXIGA_BRAND_ALL_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_FARXIGA_GENERIC_ALL_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_JARDIANCE_ALL_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_SGLT2_ALL_IN_LAST_{3,12}_MONTH` | 10 |
| SGLT2 shares, all indications | `PERC_FARXIGA_NBRX_SHARE_ALL_IN_LAST_{3,12}_MONTH`, `PERC_FARXIGA_BRAND_NBRX_SHARE_ALL_IN_LAST_{3,12}_MONTH`, `PERC_FARXIGA_GENERIC_NBRX_SHARE_ALL_IN_LAST_{3,12}_MONTH`, `PERC_JARDIANCE_NBRX_SHARE_ALL_IN_LAST_{3,12}_MONTH` | 8 |
| Other competitors, CKD scope | `TOTAL_NBRX_INVOKANA_CKD_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_STEGLATRO_CKD_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_STEROIDAL_MRA_CKD_IN_LAST_{3,12}_MONTH` | 6 |
| GLP-1, T2D and HF scopes | `TOTAL_NBRX_GLP1_T2D_IN_LAST_{3,12}_MONTH`, `PERC_GLP1_NBRX_SHARE_T2D_IN_LAST_{3,12}_MONTH`, `TOTAL_NBRX_GLP1_HF_IN_LAST_{3,12}_MONTH`, `PERC_GLP1_NBRX_SHARE_HF_IN_LAST_{3,12}_MONTH` | 8 |

**Business rules**

1. **Source:** LAAD-based LRx V7 (`kerendia_laad_based_lrx_v7`), column `NBRX`, keyed on `PROVIDER_ID`.
2. **Drug groups (V7 `PRODUCT_GROUP`):** `FARXIGA_BRAND` = FARXIGA BRAND; `FARXIGA_GENERIC` = FARXIGA GENERIC; `FARXIGA` = brand + generic; `JARDIANCE`, `INVOKANA`, `STEGLATRO`; `STEROIDAL_MRA` = every product with `PRODUCT_CLASS = 'sMRA'` (spironolactone, eplerenone and their brands); `GLP1` = Ozempic, Wegovy, Wegovy HD, Rybelsus, Mounjaro, Zepbound.
3. **Indication scopes (`INDICATION_GROUP`):** `CKD` = contains CKD (CKD, T2D + CKD, T2D + CKD + HF, CKD + HF); `T2D` = contains T2D; `HF` = contains HF; `ALL` = no filter.
4. **SGLT2 denominator:** Farxiga (brand + generic) + Jardiance + Invokana + Steglatro in the same scope and window (as NBA 3.0).
5. **Kerendia is the target** and is never used as a competitor feature.
6. **No future data:** each cohort uses LRx up to the end of its `lookback_end_date` month. 3M / 12M windows are calendar months ending with that month. No running totals.
7. **Null policy:** `TOTAL_*` = 0 when the HCP has no NBRx; `PERC_*` is empty when the denominator is 0.
8. **Reading rule (D8):** a high CKD-scoped SGLT2 share is expected to be positive for Kerendia; review a model that learns the opposite.
9. **Universe and cohorts:** every HCP in `HCP_Universe_without_call_hcps` gets one row per cohort; `cohorts/lrx` is used when it exists, else the 28 claims cohorts.

**Business logic (steps)**

1. Read LRx V7, set month = `trunc(PERIOD_ID, 'month')`, keep the competitor product groups and universe HCPs, for months from the first cohort's window start to the last cohort's lookback end.
2. Print the V1 coverage check (share of NBRx in `OTHERS` / no indication and in CKD scope, per drug, last 12 months).
3. Aggregate monthly NBRx per HCP for every drug x scope combination.
4. Join the monthly table to the cohorts on month <= lookback end and month >= lookback end - 11, then sum the 3-month and 12-month windows.
5. Build counts and shares, cross-join the universe with all cohorts, fill `TOTAL_*` with 0, add `HCP_COHORT_ID` and write Delta to `hcp_features/competitor_nbrx_share`.
6. Validate: row count, unique keys, shares in [0, 1], counts not negative, Farxiga = brand + generic, CKD <= ALL, 3M <= 12M, and a direct recompute of Jardiance CKD 12M from the LRx table.